# E3C — a stronger classical representation baseline: folklore 2-WL

The external review's third priority: the constant-feature GIN/GCN rows
are negative controls demonstrating expected 1-WL collapse, not
competitive baselines, so the paper needs one representation that exceeds
ordinary 1-WL. This notebook implements the review's preferred option:
2-WL stable-color histograms probed with the verbatim ridge protocol.

**A terminology trap, resolved up front.** "2-WL" in the Morris/GNN
convention (oblivious pair refinement) is equivalent in distinguishing
power to plain 1-WL, which Proposition 8 collapses on our censuses. The
baseline the review intends is the **folklore 2-WL**: color all ordered
vertex pairs, refine by c'(u,v) = hash(c(u,v), multiset over w of
(c(u,w), c(w,v))), iterate to stabilization. Folklore 2-WL is equivalent
to oblivious 3-WL, strictly exceeds 1-WL, and determines the adjacency
spectrum, so it is a genuinely stronger representation and the
cospectral-separation cell is a live question. The feature vector is the
histogram of stable pair colors over a vocabulary synchronized across the
whole census, so color identifiers are comparable across graphs.

**Pre-registration.** Folklore 2-WL is expected to be strong on these
censuses and may saturate the cycle rows; that outcome would not damage
the paper. Per the review's own framing, the comparison axes are: what
each representation distinguishes, which invariants are linearly
accessible, representation dimension, and whether cospectral pairs remain
separated. If 2-WL solves every target, the QuIC contribution remains its
quantum construction, fixed training-free nature, and interpretable
probability-rank hierarchy. No superior-expressive-power claim is made in
either direction. Cubic graphs at these orders are expected to be
2-WL-distinguishable (the hard cases are strongly regular graphs), stated
as an expectation and measured below, not assumed.

**Two feature arms, and why both exist.** The stable-color histogram is
maximally distinguishing and, on these censuses, useless for linear
transfer: the n = 14 prototype stabilized in 5 rounds with a 56,708-color
vocabulary, produced 509 distinct histograms of 509 graphs, and every
ridge probe on it sat at the constant-predictor floor, because
graph-unique colors give train and test rows no shared coordinates.
Maximal refinement and linear transferability trade off directly. The
primary baseline is therefore the established WL-kernel construction
(Shervashidze et al.): the concatenation of per-round histograms, whose
early rounds hold the shared, motif-carrying colors (n = 14 per-round
vocabularies: 3, 8, 1894, 56352, 56708). On the n = 14 prototype the
cumulative arm recovers C3 at 0.998, C5 at 0.917, and diameter at 0.779.
The stable-only arm stays in the table as a measured exhibit of the
tradeoff. Pre-registered cell for the real run: cumulative 2-WL's
diameter (0.779 at n = 14) exceeds QuIC's recorded 0.548, and whether
that ordering holds at n = 16 is a finding either way.

**The 1-WL row is provably constant and is asserted, not fitted.** On a
fixed-degree regular census every vertex has the same initial color and
receives the same neighbor multiset at every round, so the vertex-color
histogram is the constant vector (n) for every graph. The notebook runs
the refinement, asserts single-color stability, and reports the row as
the floor without spending a ridge fit.

**Reference columns (option a, per Luke).** The QuIC column quotes the
recorded E2 numbers as labeled constants; nothing is recomputed. The
spectral column is identity-derived where the identities decide it (C3,
C4, C5 are exactly linear in trace moments on cubic, so the spectral
reference is 1.000 by identity) and marked pending E2S for C6, diameter,
and spectral gap, whose measured spectral cells have not landed. The
2-WL column is the only measured column in this notebook, probed on
splits regenerated from seed 0 (the E7 precedent: identical to the frozen
E2 splits), with the histogram probed raw, matching the treatment of the
QuIC vector everywhere else in the study.

**Runtime.** Refinement is minutes (n = 16 projects to a few minutes from
the n = 14 timing). The ridge on the n = 16 sparse histogram matrix is
the long pole, on the order of one to three hours. RUN_NS is the session
knob; results checkpoint per census.

## Environment

In [1]:
import pickle
import time

from collections import defaultdict
from itertools import combinations

import numpy as np
import networkx as nx
import scipy.sparse as sparse

from numpy.linalg import eigvalsh

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from tqdm.notebook import tqdm


from scipy.linalg import LinAlgWarning
import warnings
warnings.filterwarnings('ignore', category=LinAlgWarning)

In [2]:
SEED = 0
RIDGE_ALPHAS = np.logspace(-14, 2, 17)
NUM_FOLDS = 5
MAX_REFINEMENT_ROUNDS = 30

CENSUS_BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}
EXPECTED_COSPECTRAL = {14: {'groups': 3, 'members': 6},
                       16: {'groups': 41, 'members': 83}}
TARGET_NAMES = ('C3', 'C4', 'C5', 'C6', 'girth', 'diameter', 'spectral_gap')

# Reference columns, option (a): recorded constants, not recomputed here.
# QuIC values are the E2 results of record (full-vector RidgeCV, frozen
# KFold(5, seed 0) splits). Spectral values marked 'identity' are exactly
# linear in trace moments on cubic graphs (C3 = tr(A^3)/6,
# C4 = (tr(A^4) - 15n)/8, C5 = (tr(A^5) - 10 tr(A^3))/10), so the spectral
# reference is 1.000 by identity; 'pending' cells await the E2S run.
REFERENCE_QUIC_R2 = {
    14: {'C3': 1.000, 'C4': 0.998, 'C5': 0.928, 'C6': 0.485,
         'girth': 0.993, 'diameter': 0.548, 'spectral_gap': 0.944},
    16: {'C3': 1.000, 'C4': 1.000, 'C5': 0.982, 'C6': 0.642,
         'girth': 0.999, 'diameter': 0.737, 'spectral_gap': 0.932},
}
REFERENCE_SPECTRAL = {'C3': 'identity 1.000', 'C4': 'identity 1.000',
                      'C5': 'identity 1.000', 'C6': 'pending E2S',
                      'girth': 'pending E2S', 'diameter': 'pending E2S',
                      'spectral_gap': 'identity 1.000'}

# Session knob: which censuses this session runs.
RUN_NS = (14, 16)

## Load the censuses and compute identity-verified targets

Only adjacency matrices are consumed; QuIC vectors play no role in this
notebook. The seven targets are recomputed in-notebook by the sanctioned
methods (cycle counts via nx.simple_cycles, metric targets from one
all-pairs shortest-path pass, spectrum via eigvalsh), and the two cubic
trace identities are asserted on every graph, so the targets are
identity-verified rather than schema-dependent.

In [3]:
CENSUS = {}
for num_vertices in RUN_NS:
    with open(CENSUS_BASES[num_vertices]
              + f"n{num_vertices}_data_dict.pkl", 'rb') as census_file:
        census_data = pickle.load(census_file)
    graph_keys = sorted(census_data)
    assert len(graph_keys) == EXPECTED_CUBIC_COUNTS[num_vertices]
    adjacency_matrices = np.stack(
        [census_data[graph_key]['adj_mat'] for graph_key in graph_keys]
    ).astype(np.int64)
    CENSUS[num_vertices] = {'adjacency_matrices': adjacency_matrices}
    print(f'n={num_vertices}: {len(graph_keys)} graphs loaded')


def compute_targets(adjacency_matrix):
    """Seven targets by the sanctioned methods; cycle counts via
    nx.simple_cycles with length_bound (the retired A^2 formula
    double-counted and is never used)."""
    graph = nx.from_numpy_array(adjacency_matrix)
    cycle_counter = defaultdict(int)
    for cycle_vertices in nx.simple_cycles(graph, length_bound=6):
        cycle_counter[len(cycle_vertices)] += 1
    shortest_paths = dict(nx.all_pairs_shortest_path_length(graph))
    eccentricities = [max(shortest_paths[source].values())
                      for source in shortest_paths]
    spectrum = np.sort(eigvalsh(adjacency_matrix.astype(np.float64)))
    return {'C3': cycle_counter[3], 'C4': cycle_counter[4],
            'C5': cycle_counter[5], 'C6': cycle_counter[6],
            'girth': nx.girth(graph),
            'diameter': max(eccentricities),
            'spectral_gap': spectrum[-1] - spectrum[-2]}


for num_vertices in RUN_NS:
    adjacency_matrices = CENSUS[num_vertices]['adjacency_matrices']
    target_rows = []
    for adjacency_matrix in tqdm(adjacency_matrices,
                                 desc=f'n={num_vertices} targets'):
        target_rows.append(compute_targets(adjacency_matrix))
    target_arrays = {target_name: np.array([row[target_name]
                                            for row in target_rows],
                                           dtype=float)
                     for target_name in TARGET_NAMES}

    # cubic trace identities, asserted on every graph
    triangle_counts = target_arrays['C3'].astype(np.int64)
    square_counts = target_arrays['C4'].astype(np.int64)
    cubed_traces = np.trace(adjacency_matrices @ adjacency_matrices
                            @ adjacency_matrices, axis1=1, axis2=2)
    fourth_traces = np.trace(np.stack(
        [np.linalg.matrix_power(adjacency_matrix, 4)
         for adjacency_matrix in adjacency_matrices]), axis1=1, axis2=2)
    assert np.array_equal(cubed_traces, 6 * triangle_counts)
    assert np.array_equal(fourth_traces,
                          8 * square_counts + 15 * num_vertices)
    CENSUS[num_vertices]['target_arrays'] = target_arrays
    print(f'n={num_vertices}: targets computed, both trace identities exact')

n=14: 509 graphs loaded
n=16: 4060 graphs loaded


n=14 targets:   0%|          | 0/509 [00:00<?, ?it/s]

n=14: targets computed, both trace identities exact


n=16 targets:   0%|          | 0/4060 [00:00<?, ?it/s]

n=16: targets computed, both trace identities exact


## The 1-WL floor — asserted, not fitted

Vertex refinement runs for real and must stabilize at a single color on
every graph of a fixed-degree regular census. The histogram row is
therefore the constant vector (n), and its R² floor is reported without a
ridge fit.

In [4]:
for num_vertices in RUN_NS:
    for adjacency_matrix in CENSUS[num_vertices]['adjacency_matrices']:
        vertex_colors = np.zeros(num_vertices, dtype=np.int64)
        for _ in range(3):   # constant colors reproduce themselves immediately
            neighbor_signatures = [
                (vertex_colors[vertex_index],
                 tuple(sorted(vertex_colors[np.nonzero(
                     adjacency_matrix[vertex_index])[0]])))
                for vertex_index in range(num_vertices)]
            assert len(set(neighbor_signatures)) == 1, (
                '1-WL produced more than one vertex color on a regular graph')
    print(f'n={num_vertices}: 1-WL constant on every graph — histogram is '
          f'({num_vertices}) for the whole census; R^2 floor is the '
          f'constant-predictor 0.0 row, no fits spent')

n=14: 1-WL constant on every graph — histogram is (14) for the whole census; R^2 floor is the constant-predictor 0.0 row, no fits spent
n=16: 1-WL constant on every graph — histogram is (16) for the whole census; R^2 floor is the constant-predictor 0.0 row, no fits spent


## Folklore 2-WL — synchronized refinement, sparse histograms

All graphs of a census refine in lockstep against one shared signature
dictionary per round, so color identifiers are comparable across graphs.
The refinement is injective round over round, so a round that adds no new
colors proves stability. The prototype validated this implementation on
the classic hexagon-versus-two-triangles pair (1-WL-equivalent, separated
here in three rounds) and on permutation invariance.

In [5]:
def histogram_block(color_snapshot, vocabulary_size, graph_count):
    """Sparse histogram matrix for one round's color arrays."""
    matrix_rows, matrix_columns, matrix_counts = [], [], []
    for graph_position, colors in enumerate(color_snapshot):
        color_ids, color_counts = np.unique(colors, return_counts=True)
        matrix_rows.extend([graph_position] * len(color_ids))
        matrix_columns.extend(color_ids.tolist())
        matrix_counts.extend(color_counts.tolist())
    return sparse.csr_matrix(
        (matrix_counts, (matrix_rows, matrix_columns)),
        shape=(graph_count, vocabulary_size), dtype=np.float64)


def folklore_2wl_census(adjacency_list, max_rounds=MAX_REFINEMENT_ROUNDS):
    """Folklore 2-WL over one census with a shared vocabulary.

    Pair colors start as {diagonal, edge, non-edge}. Each round encodes,
    for every ordered pair (u, v), the multiset over w of the composition
    (color(u, w), color(w, v)) together with the pair's old color, and maps
    that signature to a global integer through one dictionary shared by all
    graphs in the round. Returns a dict with the cumulative all-rounds
    feature matrix (the WL-kernel construction: per-round histogram blocks
    concatenated with disjoint vocabularies), the stable-only histogram
    matrix (final round), the per-round vocabulary sizes, and the rounds
    used.
    """
    graph_count = len(adjacency_list)
    vertex_count = adjacency_list[0].shape[0]
    color_arrays = []
    for adjacency in adjacency_list:
        initial_colors = np.where(adjacency > 0, 1, 2).astype(np.int64)
        np.fill_diagonal(initial_colors, 0)
        color_arrays.append(initial_colors)
    global_color_count = 3
    round_snapshots = [[colors.copy() for colors in color_arrays]]
    round_vocabulary_sizes = [3]

    rounds_used = 0
    for round_number in range(1, max_rounds + 1):
        signature_dictionary = {}
        new_color_arrays = []
        for colors in color_arrays:
            # composition_codes[u, v, w] encodes (colors[u, w], colors[w, v])
            composition_codes = (colors[:, None, :] * global_color_count
                                 + colors.T[None, :, :])
            sorted_compositions = np.sort(composition_codes, axis=2)
            signature_block = np.concatenate(
                [colors[:, :, None], sorted_compositions], axis=2)
            flat_signatures = signature_block.reshape(
                vertex_count * vertex_count, vertex_count + 1)
            new_colors = np.empty(vertex_count * vertex_count, dtype=np.int64)
            for pair_position, signature_row in enumerate(flat_signatures):
                signature_key = signature_row.tobytes()
                if signature_key not in signature_dictionary:
                    signature_dictionary[signature_key] = (
                        len(signature_dictionary))
                new_colors[pair_position] = signature_dictionary[signature_key]
            new_color_arrays.append(
                new_colors.reshape(vertex_count, vertex_count))
        rounds_used = round_number
        new_global_color_count = len(signature_dictionary)
        color_arrays = new_color_arrays
        if new_global_color_count == global_color_count:
            break   # refinement is injective, so equal counts mean stable
        global_color_count = new_global_color_count
        round_snapshots.append([colors.copy() for colors in color_arrays])
        round_vocabulary_sizes.append(new_global_color_count)
    assert rounds_used < max_rounds, 'refinement did not stabilize — raise cap'

    cumulative_matrix = sparse.hstack(
        [histogram_block(snapshot, vocabulary_size, graph_count)
         for snapshot, vocabulary_size in zip(round_snapshots,
                                              round_vocabulary_sizes)],
        format='csr')
    stable_matrix = histogram_block(round_snapshots[-1],
                                    round_vocabulary_sizes[-1], graph_count)
    return {'cumulative_matrix': cumulative_matrix,
            'stable_matrix': stable_matrix,
            'round_vocabulary_sizes': round_vocabulary_sizes,
            'rounds_used': rounds_used}


for num_vertices in RUN_NS:
    start_time = time.time()
    refinement = folklore_2wl_census(
        list(CENSUS[num_vertices]['adjacency_matrices']))
    CENSUS[num_vertices]['refinement'] = refinement
    stable_matrix = refinement['stable_matrix']
    distinct_histogram_count = len({
        stable_matrix.getrow(row_index).todense().tobytes()
        for row_index in range(stable_matrix.shape[0])})
    print(f"n={num_vertices}: per-round vocabularies "
          f"{refinement['round_vocabulary_sizes']} "
          f"(cumulative {refinement['cumulative_matrix'].shape[1]}), "
          f"{refinement['rounds_used']} rounds, {time.time()-start_time:.0f}s; "
          f"distinct stable histograms "
          f"{distinct_histogram_count}/{stable_matrix.shape[0]}")

n=14: per-round vocabularies [3, 8, 1894, 56352, 56708] (cumulative 114965), 5 rounds, 1s; distinct stable histograms 509/509
n=16: per-round vocabularies [3, 8, 2230, 741367, 757353] (cumulative 1500961), 5 rounds, 43s; distinct stable histograms 4060/4060


## Cospectral separation — exact integer comparison

The exact int64 trace census is recomputed (the E8A method, asserted
against its counts), and every within-group pair's 2-WL histograms are
compared as exact integer vectors. No tolerance is involved: histograms
are either identical or distinct. The QuIC row of this comparison is the
E2C record — all 46 pairs separate — and the spectrum ties everywhere by
construction.

In [6]:
COSPECTRAL_SEPARATION = {}
for num_vertices in RUN_NS:
    adjacency_matrices = CENSUS[num_vertices]['adjacency_matrices']
    census_size = len(adjacency_matrices)
    trace_tuples = np.zeros((census_size, num_vertices), dtype=np.int64)
    walk_matrix = adjacency_matrices.copy()
    trace_tuples[:, 0] = np.trace(walk_matrix, axis1=1, axis2=2)
    for power_index in range(1, num_vertices):
        walk_matrix = walk_matrix @ adjacency_matrices
        trace_tuples[:, power_index] = np.trace(walk_matrix, axis1=1, axis2=2)
    groups_by_tuple = defaultdict(list)
    for graph_index in range(census_size):
        groups_by_tuple[tuple(trace_tuples[graph_index])].append(graph_index)
    cospectral_groups = sorted(sorted(member_list) for member_list
                               in groups_by_tuple.values()
                               if len(member_list) > 1)
    member_count = len({graph_index for group in cospectral_groups
                        for graph_index in group})
    assert len(cospectral_groups) == EXPECTED_COSPECTRAL[num_vertices]['groups']
    assert member_count == EXPECTED_COSPECTRAL[num_vertices]['members']

    histogram_matrix = CENSUS[num_vertices]['refinement']['stable_matrix']
    separated_pairs, tied_pairs = [], []
    for group in cospectral_groups:
        for first_index, second_index in combinations(group, 2):
            first_histogram = histogram_matrix.getrow(first_index)
            second_histogram = histogram_matrix.getrow(second_index)
            if (first_histogram - second_histogram).nnz == 0:
                tied_pairs.append((first_index, second_index))
            else:
                separated_pairs.append((first_index, second_index))
    COSPECTRAL_SEPARATION[num_vertices] = {
        'separated': separated_pairs, 'tied': tied_pairs}
    print(f'n={num_vertices}: 2-WL separates {len(separated_pairs)} of '
          f'{len(separated_pairs) + len(tied_pairs)} cospectral pairs '
          f'(exact integer comparison)'
          + (f'; TIED: {tied_pairs}' if tied_pairs else ''))

n=14: 2-WL separates 3 of 3 cospectral pairs (exact integer comparison)
n=16: 2-WL separates 43 of 43 cospectral pairs (exact integer comparison)


## Ridge probe — the verbatim protocol on raw histograms

RidgeCV(alphas = logspace(-14, 2, 17), cv = 5) on KFold(5, shuffle,
seed 0) splits regenerated from seed (the E7 precedent: identical to the
frozen E2 splits). The histogram is probed raw, matching the treatment of
the QuIC vector everywhere else in the study. Results checkpoint per
census.

In [7]:
FEATURE_ARMS = ('2wl_all_rounds', '2wl_stable_only')

RESULTS = {}
for num_vertices in RUN_NS:
    arm_matrices = {
        '2wl_all_rounds': CENSUS[num_vertices]['refinement']['cumulative_matrix'],
        '2wl_stable_only': CENSUS[num_vertices]['refinement']['stable_matrix']}
    fold_generator = KFold(n_splits=NUM_FOLDS, shuffle=True,
                           random_state=SEED)
    fold_splits = list(fold_generator.split(
        np.arange(arm_matrices['2wl_all_rounds'].shape[0])))

    RESULTS[num_vertices] = {}
    for target_name in tqdm(TARGET_NAMES, desc=f'n={num_vertices} probes'):
        target_values = CENSUS[num_vertices]['target_arrays'][target_name]
        RESULTS[num_vertices][target_name] = {}
        for arm_name in FEATURE_ARMS:
            feature_matrix = arm_matrices[arm_name]
            fold_scores, chosen_alphas = [], []
            for train_indices, test_indices in fold_splits:
                probe_model = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
                probe_model.fit(feature_matrix[train_indices],
                                target_values[train_indices])
                fold_scores.append(r2_score(
                    target_values[test_indices],
                    probe_model.predict(feature_matrix[test_indices])))
                chosen_alphas.append(probe_model.alpha_)
            fold_scores = np.array(fold_scores)
            RESULTS[num_vertices][target_name][arm_name] = {
                'r2_folds': fold_scores,
                'r2_mean': float(fold_scores.mean()),
                'r2_sd': float(fold_scores.std()),
                'alphas_chosen': chosen_alphas}
        all_rounds_row = RESULTS[num_vertices][target_name]['2wl_all_rounds']
        stable_row = RESULTS[num_vertices][target_name]['2wl_stable_only']
        print(f"  {target_name}: all-rounds "
              f"{all_rounds_row['r2_mean']:+.3f}±{all_rounds_row['r2_sd']:.3f}  "
              f"stable-only {stable_row['r2_mean']:+.3f}±{stable_row['r2_sd']:.3f}")

    with open('/kaggle/working/e3c_2wl_baseline.pkl', 'wb') as output_file:
        pickle.dump({'results': RESULTS,
                     'round_vocabulary_sizes':
                         {n: CENSUS[n]['refinement']['round_vocabulary_sizes']
                          for n in RESULTS},
                     'cospectral_separation': COSPECTRAL_SEPARATION,
                     'reference_quic_r2': REFERENCE_QUIC_R2,
                     'reference_spectral': REFERENCE_SPECTRAL,
                     'seed': SEED, 'alphas_grid': RIDGE_ALPHAS,
                     'run_ns': RUN_NS}, output_file)
    print(f'n={num_vertices} complete — checkpointed to e3c_2wl_baseline.pkl')

n=14 probes:   0%|          | 0/7 [00:00<?, ?it/s]

  C3: all-rounds +0.998±0.001  stable-only -0.005±0.003
  C4: all-rounds +0.993±0.004  stable-only -0.025±0.022
  C5: all-rounds +0.917±0.038  stable-only -0.011±0.013
  C6: all-rounds +0.879±0.061  stable-only -0.004±0.002
  girth: all-rounds +0.787±0.071  stable-only -0.020±0.013
  diameter: all-rounds +0.779±0.031  stable-only -0.010±0.011
  spectral_gap: all-rounds +0.947±0.028  stable-only -0.004±0.004
n=14 complete — checkpointed to e3c_2wl_baseline.pkl


n=16 probes:   0%|          | 0/7 [00:00<?, ?it/s]

  C3: all-rounds +1.000±0.000  stable-only -0.007±0.007
  C4: all-rounds +0.999±0.000  stable-only -0.018±0.009
  C5: all-rounds +0.993±0.002  stable-only -0.001±0.001
  C6: all-rounds +0.983±0.003  stable-only -0.002±0.001
  girth: all-rounds +0.844±0.016  stable-only -0.006±0.002
  diameter: all-rounds +0.861±0.012  stable-only -0.010±0.007
  spectral_gap: all-rounds +0.982±0.002  stable-only -0.014±0.016
n=16 complete — checkpointed to e3c_2wl_baseline.pkl


## Comparison table

The 2-WL column is measured here; the QuIC column quotes the E2 record;
the spectral column is identity-derived where the identities decide it
and marked pending E2S elsewhere; the 1-WL row is the asserted constant
floor. The dimension line reports the review's fourth comparison axis.

In [8]:
for num_vertices in RUN_NS:
    print(f'\n=== n={num_vertices} — representation comparison ===')
    refinement = CENSUS[num_vertices]['refinement']
    print(f'dimensions: QuIC 2^{num_vertices} = {1 << num_vertices}; '
          f'spectrum {num_vertices}; '
          f"2-WL cumulative {refinement['cumulative_matrix'].shape[1]} "
          f"(per round {refinement['round_vocabulary_sizes']}); "
          f"2-WL stable {refinement['stable_matrix'].shape[1]}; "
          f"1-WL 1 (constant)")
    separation = COSPECTRAL_SEPARATION[num_vertices]
    total_pairs = len(separation['separated']) + len(separation['tied'])
    print(f'cospectral pairs separated: QuIC {total_pairs}/{total_pairs} '
          f'(E2C record); 2-WL {len(separation["separated"])}/{total_pairs} '
          f'(exact, this run); spectrum 0/{total_pairs} (by construction); '
          f'1-WL 0/{total_pairs} (constant)')
    print(f'{"target":>13} | {"2WL all-rounds":>16} {"2WL stable-only":>16} '
          f'| {"QuIC (E2 record)":>16} | {"spectral (reference)":>20}')
    for target_name in TARGET_NAMES:
        all_rounds_row = RESULTS[num_vertices][target_name]['2wl_all_rounds']
        stable_row = RESULTS[num_vertices][target_name]['2wl_stable_only']
        print(f"{target_name:>13} | "
              f"{all_rounds_row['r2_mean']:+.3f}±{all_rounds_row['r2_sd']:.3f}"
              f"{'':>4} "
              f"{stable_row['r2_mean']:+.3f}±{stable_row['r2_sd']:.3f}{'':>4} "
              f"| {REFERENCE_QUIC_R2[num_vertices][target_name]:+16.3f} "
              f"| {REFERENCE_SPECTRAL[target_name]:>20}")


=== n=14 — representation comparison ===
dimensions: QuIC 2^14 = 16384; spectrum 14; 2-WL cumulative 114965 (per round [3, 8, 1894, 56352, 56708]); 2-WL stable 56708; 1-WL 1 (constant)
cospectral pairs separated: QuIC 3/3 (E2C record); 2-WL 3/3 (exact, this run); spectrum 0/3 (by construction); 1-WL 0/3 (constant)
       target |   2WL all-rounds  2WL stable-only | QuIC (E2 record) | spectral (reference)
           C3 | +0.998±0.001     -0.005±0.003     |           +1.000 |       identity 1.000
           C4 | +0.993±0.004     -0.025±0.022     |           +0.998 |       identity 1.000
           C5 | +0.917±0.038     -0.011±0.013     |           +0.928 |       identity 1.000
           C6 | +0.879±0.061     -0.004±0.002     |           +0.485 |          pending E2S
        girth | +0.787±0.071     -0.020±0.013     |           +0.993 |          pending E2S
     diameter | +0.779±0.031     -0.010±0.011     |           +0.548 |          pending E2S
 spectral_gap | +0.947±0.028     -0.004

## Results

(Written after the run. The reading order: (1) the identity asserts, the
1-WL constancy asserts, and the E8A census asserts, which passing
silently licenses the table; (2) the cospectral-separation line, the one
cell where 2-WL could genuinely differ from both QuIC and the spectrum —
the prototype's 509/509 distinct histograms at n = 14 makes full
separation the expected reading, and any tied pair is a finding worth a
sentence; (3) the R² columns read against the pre-registration, with the two arms
read together: the all-rounds arm is the honest baseline, the stable-only
arm exhibits the refinement-versus-transfer tradeoff (graph-unique colors
distinguish everything and transfer nothing), and the diameter cell
checks whether cumulative 2-WL's n = 14 win over QuIC (0.779 vs 0.548)
holds at n = 16; 2-WL saturating the cycle rows is the expected strong
outcome and costs the paper nothing, per the review's own framing — the axes of the comparison
are distinguishing power, linear accessibility, dimension, and cospectral
separation, and the QuIC contribution named in the review survives every
outcome; (4) the dimension line: the 2-WL vocabulary against 2^n and
against n. Verb discipline: 2-WL "linearly decodes" or "does not linearly
decode"; no representation "wins" outside the stated axes; no
expressive-power claims beyond what the separation cell shows.)